# Lecture 15 — Implementing Convolutions

**CMU 10-414/714, Deep Learning Systems**
Companion notebook · Sangram Lembe

The lecture implements the same convolution four ways, each faster than the last, and checks every
one against a trusted reference:

1. seven nested loops
2. a $1\times1$ convolution as one matrix multiply
3. a $K\times K$ convolution as $K^2$ shifted matrix multiplies
4. im2col as **one** matrix multiply, built with `as_strided` and no loops at all

Kolter uses PyTorch as the reference. PyTorch is not installed here, so the reference is SciPy's
`correlate2d` — an independent implementation, used exactly the same way.

In [1]:
import numpy as np, time
from numpy.lib.stride_tricks import as_strided
from scipy.signal import correlate2d

rng = np.random.default_rng(0)

def timeit(fn, repeats=3):
    fn(); best = float("inf")
    for _ in range(repeats):
        t = time.perf_counter(); fn(); best = min(best, time.perf_counter() - t)
    return best

## 0. Storage order, and a reference to test against

Images are stored **NHWC** (batch, height, width, channels) and filters as
$K\times K\times C_{in}\times C_{out}$. With channels last, each pixel's channel vector is contiguous —
look at the strides:

In [2]:
Z = rng.standard_normal((2, 5, 5, 3)).astype(np.float32)
print("NHWC strides (in elements):", tuple(s // 4 for s in Z.strides), " <- channel stride is 1")
Z_nchw = np.ascontiguousarray(Z.transpose(0, 3, 1, 2))
print("NCHW strides (in elements):", tuple(s // 4 for s in Z_nchw.strides), "<- a pixel's channels are 25 apart")

NHWC strides (in elements): (75, 15, 3, 1)  <- channel stride is 1
NCHW strides (in elements): (75, 25, 5, 1) <- a pixel's channels are 25 apart


In [3]:
def conv_reference(Z, W):
    """Independent reference (no padding): sum over input channels of 2-D correlations.
    Z: (N, H, W, Cin).  W: (K, K, Cin, Cout)."""
    N, H, Wd, Cin = Z.shape
    K, _, _, Cout = W.shape
    out = np.zeros((N, H - K + 1, Wd - K + 1, Cout), dtype=np.float64)
    for n in range(N):
        for co in range(Cout):
            for ci in range(Cin):
                out[n, :, :, co] += correlate2d(Z[n, :, :, ci], W[:, :, ci, co], mode="valid")
    return out

Z = rng.standard_normal((4, 12, 12, 3)).astype(np.float32)
W = rng.standard_normal((3, 3, 3, 8)).astype(np.float32)
ref = conv_reference(Z, W)
print("input", Z.shape, "filter", W.shape, "-> output", ref.shape, " (no padding: 12 - 3 + 1 = 10)")

input (4, 12, 12, 3) filter (3, 3, 3, 8) -> output (4, 10, 10, 8)  (no padding: 12 - 3 + 1 = 10)


## 1. Seven nested loops

In [4]:
def conv_loops(Z, W):
    N, H, Wd, Cin = Z.shape
    K, _, _, Cout = W.shape
    out = np.zeros((N, H - K + 1, Wd - K + 1, Cout))
    for n in range(N):
        for y in range(H - K + 1):
            for x in range(Wd - K + 1):
                for co in range(Cout):
                    for i in range(K):
                        for j in range(K):
                            for ci in range(Cin):
                                out[n, y, x, co] += Z[n, y + i, x + j, ci] * W[i, j, ci, co]
    return out

out1 = conv_loops(Z, W)
print("matches reference:", np.allclose(out1, ref, atol=1e-4))

matches reference: True


## 2. A $1\times1$ convolution is one matrix multiply

With a $1\times1$ filter, each output pixel is that pixel's channel vector times a
$C_{in}\times C_{out}$ matrix. For NHWC data `Z @ W` does exactly this — NumPy treats all leading
dimensions as rows.

In [5]:
W1 = rng.standard_normal((1, 1, 3, 8)).astype(np.float32)
direct   = Z @ W1[0, 0]                                             # (N,H,W,Cin) @ (Cin,Cout)
reshaped = (Z.reshape(-1, 3) @ W1[0, 0]).reshape(4, 12, 12, 8)      # same thing, explicitly
print("Z @ W == reshape version :", np.allclose(direct, reshaped))
print("matches reference        :", np.allclose(direct, conv_reference(Z, W1), atol=1e-4))

Z @ W == reshape version : True
matches reference        : True


## 3. A $K\times K$ convolution is $K^2$ shifted matrix multiplies

A $3\times3$ convolution is nine $1\times1$ convolutions, each applied to the image shifted by one filter
position, added together. Seven loops become two.

In [6]:
def conv_matmul(Z, W):
    N, H, Wd, Cin = Z.shape
    K, _, _, Cout = W.shape
    out = np.zeros((N, H - K + 1, Wd - K + 1, Cout))
    for i in range(K):
        for j in range(K):
            out += Z[:, i:i + H - K + 1, j:j + Wd - K + 1, :] @ W[i, j]
    return out

print("matches reference:", np.allclose(conv_matmul(Z, W), ref, atol=1e-4))

matches reference: True


## 4. `as_strided`, on the lecture's tiling example

`as_strided` gives an array's memory a new shape and new strides without copying. Split a $6\times6$
matrix into $2\times2$ tiles: a plain reshape gets the wrong tiles, the right strides get the right ones.

In [7]:
A = np.arange(36, dtype=np.float32).reshape(6, 6)
e = A.itemsize                                                      # NumPy strides are in BYTES

print("reshape(3,3,2,2) tile 0 :", A.reshape(3, 3, 2, 2)[0, 0].ravel().tolist(), " <- wrong")
T = as_strided(A, shape=(3, 3, 2, 2), strides=np.array([12, 2, 6, 1]) * e)
print("as_strided tile 0       :", T[0, 0].ravel().tolist(), " <- right")
print("as_strided tile (0,1)   :", T[0, 1].ravel().tolist())
print("\nshares memory with A    :", np.shares_memory(T, A))
C = np.ascontiguousarray(T)
print("after ascontiguousarray :", C.ravel()[:8].tolist(), "...  shares memory:", np.shares_memory(C, A))

reshape(3,3,2,2) tile 0 : [0.0, 1.0, 2.0, 3.0]  <- wrong
as_strided tile 0       : [0.0, 1.0, 6.0, 7.0]  <- right
as_strided tile (0,1)   : [2.0, 3.0, 8.0, 9.0]

shares memory with A    : True
after ascontiguousarray : [0.0, 1.0, 6.0, 7.0, 2.0, 3.0, 8.0, 9.0] ...  shares memory: False


Because nothing is copied, a write through the view changes the original — the reason NumPy hides this
function so deep. Try it on a copy:

In [8]:
A2 = A.copy()
T2 = as_strided(A2, shape=(3, 3, 2, 2), strides=np.array([12, 2, 6, 1]) * e)
T2[0, 0, 1, 1] = -99.0                       # write into "tile 0"...
print("A2[1, 1] is now", A2[1, 1], "  <- ...and the original changed")

A2[1, 1] is now -99.0   <- ...and the original changed


## 5. im2col with `as_strided`: overlapping windows

For a $6\times6$ image and a $3\times3$ filter, strides $(6, 1, 6, 1)$ give a $4\times4\times3\times3$ view —
each output position's patch. Unlike tiles, these windows **overlap**, so turning the view into a matrix
must copy, and it grows.

In [9]:
Wf = np.arange(9, dtype=np.float32).reshape(3, 3)
B = as_strided(A, shape=(4, 4, 3, 3), strides=np.array([6, 1, 6, 1]) * e)   # free view
M = B.reshape(16, 9)                                                         # this copies
out = (M @ Wf.reshape(9)).reshape(4, 4)
print("matches correlate2d:", np.allclose(out, correlate2d(A, Wf, mode="valid")))
print(f"numbers in the image : {A.size}")
print(f"numbers in the view  : {A.size}  (shares memory: {np.shares_memory(B, A)})")
print(f"numbers in the matrix: {M.size}  (shares memory: {np.shares_memory(M, A)})")

matches correlate2d: True
numbers in the image : 36
numbers in the view  : 36  (shares memory: True)
numbers in the matrix: 144  (shares memory: False)


## 6. The full batched version: one matrix multiply

The view becomes six-dimensional, and the height and width strides appear **twice** — once to move
between windows, once to move within a window.

In [10]:
def conv_im2col(Z, W):
    N, H, Wd, Cin = Z.shape
    K, _, _, Cout = W.shape
    Ns, Hs, Ws, Cs = Z.strides
    A = as_strided(Z, shape=(N, H - K + 1, Wd - K + 1, K, K, Cin),
                   strides=(Ns, Hs, Ws, Hs, Ws, Cs))            # free: just a view
    A = A.reshape(-1, K * K * Cin)                             # the copy happens here
    out = A @ W.reshape(-1, Cout)                              # ONE matrix multiply
    return out.reshape(N, H - K + 1, Wd - K + 1, Cout)

print("matches reference:", np.allclose(conv_im2col(Z, W), ref, atol=1e-4))

matches reference: True


## 7. Race them

A larger batch for the fast versions; the seven-loop version only gets a small one, because it is that
slow.

In [11]:
Zs, Ws_ = rng.standard_normal((2, 12, 12, 4)).astype(np.float32), rng.standard_normal((3, 3, 4, 8)).astype(np.float32)
Zb, Wb  = rng.standard_normal((32, 32, 32, 8)).astype(np.float32), rng.standard_normal((3, 3, 8, 16)).astype(np.float32)

t_loops_small = timeit(lambda: conv_loops(Zs, Ws_), repeats=1)
t_mm_small    = timeit(lambda: conv_matmul(Zs, Ws_))
t_mm          = timeit(lambda: conv_matmul(Zb, Wb))
t_i2c         = timeit(lambda: conv_im2col(Zb, Wb))
print("small batch (2 x 12 x 12 x 4):")
print(f"  seven loops        {t_loops_small*1e3:9.1f} ms")
print(f"  K^2 matmuls        {t_mm_small*1e3:9.3f} ms   ({t_loops_small/t_mm_small:,.0f}x faster)")
print("big batch (32 x 32 x 32 x 8):")
print(f"  K^2 matmuls        {t_mm*1e3:9.2f} ms")
print(f"  im2col, 1 matmul   {t_i2c*1e3:9.2f} ms")
print("big batch results agree:", np.allclose(conv_matmul(Zb, Wb), conv_im2col(Zb, Wb), atol=1e-3))

small batch (2 x 12 x 12 x 4):
  seven loops             34.0 ms
  K^2 matmuls            0.055 ms   (613x faster)
big batch (32 x 32 x 32 x 8):
  K^2 matmuls             4.17 ms
  im2col, 1 matmul        1.75 ms
big batch results agree: True


## 8. The price: memory

The as_strided view is free; the reshape that compacts it is not. Every pixel is copied once for each
window that contains it — up to $K^2$ times.

In [12]:
N, H, Wd, Cin = Zb.shape
K = 3
cols = as_strided(Zb, shape=(N, H - K + 1, Wd - K + 1, K, K, Cin),
                  strides=(Zb.strides[0], Zb.strides[1], Zb.strides[2],
                           Zb.strides[1], Zb.strides[2], Zb.strides[3])).reshape(-1, K * K * Cin)
print(f"input image batch : {Zb.nbytes / 1e6:6.2f} MB")
print(f"im2col matrix     : {cols.nbytes / 1e6:6.2f} MB   ({cols.nbytes / Zb.nbytes:.1f}x larger; "
      f"K^2 = {K*K}, slightly less because of the edges)")

input image batch :   1.05 MB
im2col matrix     :   8.29 MB   (7.9x larger; K^2 = 9, slightly less because of the edges)


## 9. Padding is added separately

These convolutions shrink the image by $K-1$. To keep the size, pad with $(K-1)/2$ zeros first — the
lecture keeps padding out of the convolution itself for exactly this reason.

In [13]:
p = (3 - 1) // 2
Zp = np.pad(Z, ((0, 0), (p, p), (p, p), (0, 0)))
print("input", Z.shape, "-> padded", Zp.shape, "-> output", conv_im2col(Zp, W).shape)

input (4, 12, 12, 3) -> padded (4, 14, 14, 3) -> output (4, 12, 12, 8)


## Summary

- NHWC keeps a pixel's channels contiguous (stride 1), so channels feed straight into matrix multiplies.
- All four implementations match an independent reference exactly.
- A $1\times1$ convolution is `Z @ W`; a $K\times K$ one is $K^2$ shifted matrix multiplies — hundreds of
  times faster than seven Python loops.
- `as_strided` builds tiles and im2col windows without copying — and a write through the view changed
  the original.
- Overlapping windows must be copied to become a matrix: 36 numbers became 144, and the batched im2col
  matrix was about 8× the size of the input. That copy is `compact()`.